# Fix 6 Full Self-RAG Only (Kaggle T4x2)

Use this notebook on a fresh Kaggle GPU session with Internet enabled and GPU set to T4 x2. It runs only the full Fix 6 Self-RAG path, then creates one downloadable zip.

This notebook does not run the no-Self-RAG stage and does not run the Self-RAG smoke stage.

## 1. Bootstrap repository

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working

if [ ! -d rag-hallucination-detection/.git ]; then
  git clone --progress --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git rag-hallucination-detection
else
  git -C rag-hallucination-detection fetch origin main
  git -C rag-hallucination-detection checkout main
  git -C rag-hallucination-detection pull --ff-only origin main
fi

cd rag-hallucination-detection
git log -1 --oneline
ls scripts/kaggle_fix6_t4x2.sh scripts/kaggle_stream_fix6_t4x2.py experiments/fix_06_baseline_h2h_pareto.py

## 2. Setup runtime

In [ ]:
import os
import subprocess
import time
from pathlib import Path

repo = Path('/kaggle/working/rag-hallucination-detection')
log_path = Path('/kaggle/working/fix6_visible_setup.log')

env = os.environ.copy()
env.update({
    'REPO_DIR': str(repo),
    'HEARTBEAT_SECONDS': '30',
    'FIX6_DATASETS': 'squad hotpotqa',
    'FIX6_N': '200',
    'FIX6_MAX_CONTEXTS': '250',
    'FIX6_SAVE_EVERY': '10',
    'FIX6_CUDA_VISIBLE_DEVICES': '1',
    'SELFRAG_MODEL': 'selfrag/selfrag_llama2_7b',
    'SELFRAG_QUANT': '8bit',
    'PYTHONUNBUFFERED': '1',
})

def tail(path, n=25):
    path = Path(path)
    if not path.exists():
        return '(log not created yet)'
    lines = path.read_text(errors='replace').splitlines()
    return '\n'.join(lines[-n:]) if lines else '(log is empty so far)'

# Keep this run clean even though origin/main contains the completed no-Self-RAG outputs.
subprocess.run(
    "rm -rf data/revision/fix_06 results/revision/fix_06 chroma_db_fix06 && "
    "mkdir -p data/revision/fix_06 results/revision/fix_06 logs/revision && "
    "rm -f logs/revision/fix_06_* /kaggle/working/fix6_t4x2_*.log /kaggle/working/fix6_t4x2_wrapper.log",
    cwd=repo,
    shell=True,
    check=True,
)

cmd = ['python', '-u', 'scripts/kaggle_stream_fix6_t4x2.py', '--stage', 'setup', '--heartbeat', '30']
print('[visible] START setup', flush=True)
print('[visible] full log:', log_path, flush=True)

with log_path.open('w') as log_file:
    proc = subprocess.Popen(cmd, cwd=repo, stdout=log_file, stderr=subprocess.STDOUT, text=True, env=env)
    start = time.time()
    while proc.poll() is None:
        time.sleep(30)
        elapsed = int(time.time() - start)
        print(f'\n[visible] setup still running elapsed={elapsed}s')
        print(tail(log_path, 25), flush=True)
    rc = proc.wait()

print('\n[visible] setup finished rc=', rc)
print(tail(log_path, 80), flush=True)
if rc != 0:
    raise RuntimeError(f'setup failed with rc={rc}; see {log_path}')

## 3. Run full Self-RAG

In [ ]:
import csv
import os
import subprocess
import time
from pathlib import Path

repo = Path('/kaggle/working/rag-hallucination-detection')
log_path = Path('/kaggle/working/fix6_visible_selfrag.log')

env = os.environ.copy()
env.update({
    'REPO_DIR': str(repo),
    'HEARTBEAT_SECONDS': '30',
    'FIX6_DATASETS': 'squad hotpotqa',
    'FIX6_N': '200',
    'FIX6_MAX_CONTEXTS': '250',
    'FIX6_SAVE_EVERY': '10',
    'FIX6_CUDA_VISIBLE_DEVICES': '1',
    'SELFRAG_MODEL': 'selfrag/selfrag_llama2_7b',
    'SELFRAG_QUANT': '8bit',
    'PYTHONUNBUFFERED': '1',
})

def count_rows(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return 0
    try:
        with path.open(newline='') as f:
            return max(0, sum(1 for _ in csv.reader(f)) - 1)
    except Exception:
        return '?'

def status_line():
    return (
        f"per_query={count_rows(repo / 'data/revision/fix_06/per_query.csv')} "
        f"squad_partial={count_rows(repo / 'data/revision/fix_06/per_query_squad_partial.csv')} "
        f"hotpot_partial={count_rows(repo / 'data/revision/fix_06/per_query_hotpotqa_partial.csv')} "
        f"summary={count_rows(repo / 'results/revision/fix_06/h2h_summary.csv')}"
    )

def tail(path, n=25):
    path = Path(path)
    if not path.exists():
        return '(log not created yet)'
    lines = path.read_text(errors='replace').splitlines()
    return '\n'.join(lines[-n:]) if lines else '(log is empty so far)'

cmd = ['python', '-u', 'scripts/kaggle_stream_fix6_t4x2.py', '--stage', 'selfrag', '--heartbeat', '30']
print('[visible] START full Self-RAG', flush=True)
print('[visible] full log:', log_path, flush=True)
print('[visible] expected final: per_query=1600 squad_partial=800 hotpot_partial=800 summary=8', flush=True)

with log_path.open('w') as log_file:
    proc = subprocess.Popen(cmd, cwd=repo, stdout=log_file, stderr=subprocess.STDOUT, text=True, env=env)
    start = time.time()
    while proc.poll() is None:
        time.sleep(30)
        elapsed = int(time.time() - start)
        print(f'\n[visible] selfrag running elapsed={elapsed}s {status_line()}')
        print(tail(log_path, 25), flush=True)
    rc = proc.wait()

print('\n[visible] selfrag finished rc=', rc)
print('[visible] final', status_line())
print(tail(log_path, 80), flush=True)
if rc != 0:
    raise RuntimeError(f'selfrag failed with rc={rc}; see {log_path}')

## 4. Create one download

In [ ]:
import csv
import os
import subprocess
from pathlib import Path
from IPython.display import FileLink, display

repo = Path('/kaggle/working/rag-hallucination-detection')
zip_path = Path('/kaggle/working/AAA_FIX6_FULL_SELFRAG_ONLY_OUTPUTS.zip')

def count_rows(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return 0
    with path.open(newline='') as f:
        return max(0, sum(1 for _ in csv.reader(f)) - 1)

print('per_query rows:', count_rows(repo / 'data/revision/fix_06/per_query.csv'))
print('squad partial rows:', count_rows(repo / 'data/revision/fix_06/per_query_squad_partial.csv'))
print('hotpot partial rows:', count_rows(repo / 'data/revision/fix_06/per_query_hotpotqa_partial.csv'))
print('summary rows:', count_rows(repo / 'results/revision/fix_06/h2h_summary.csv'))

if zip_path.exists():
    zip_path.unlink()

cmd = r'''
set -euo pipefail
cd /kaggle/working/rag-hallucination-detection
zip -r /kaggle/working/AAA_FIX6_FULL_SELFRAG_ONLY_OUTPUTS.zip \
  data/revision/fix_06 \
  results/revision/fix_06 \
  logs/revision/fix_06_* \
  scripts/kaggle_fix6_t4x2.sh \
  scripts/kaggle_stream_fix6_t4x2.py \
  experiments/fix_06_baseline_h2h_pareto.py \
  src/selfrag_wrapper.py \
  docs/revision/README.md \
  docs/revision/status.md \
  docs/revision/runbook.md

cd /kaggle/working
zip -r /kaggle/working/AAA_FIX6_FULL_SELFRAG_ONLY_OUTPUTS.zip \
  fix6_t4x2_logs \
  fix6_t4x2_setup.log \
  fix6_t4x2_selfrag.log \
  fix6_t4x2_wrapper.log
ls -lh /kaggle/working/AAA_FIX6_FULL_SELFRAG_ONLY_OUTPUTS.zip
'''
subprocess.run(['bash', '-lc', cmd], check=True)
display(FileLink(str(zip_path)))